In [ ]:
import sys
from pathlib import Path

API = Path.cwd().parent                     # .../StaffScheduling/api (Jupyter startet im Ordner des Notebooks)
ROOT = API.parent
sys.path[:0] = [str(API), str(API / "tests")]   # app.* und die Test-Bausteine importierbar machen

# --- Datenstrukturen (nur app.domain: braucht pydantic + holidays, keine Datenbank, kein OR-Tools) ---
from app.domain import (
    POLICY,                 # Regelparameter: min_rest_minutes, balance_tolerance_minutes, max_consecutive_nights, ...
    SchedulingDataset,      # die komplette Eingabe eines Monats
    Assignment,             # ein Dienst (Ergebnis): employee_id, date, planning_unit_id, shift_id, staff_level
    Gap,                    # eine Lücke (Ergebnis)
    DemandRequirement, Shift, ShiftType, StaffLevel, AvailabilityType, WishType, FREE_WISHES,
    duty_times,             # echte Start-/Endzeiten eines Dienstes (inkl. Sommerzeit)
    is_working_day,         # Werktag ja/nein
    check_schedule,         # unabhängiger Schedule-Check: Regeln + Kennzahlen
)
from app.domain.rules import BLOCKING_AVAILABILITY, APPROVED_FREE
from app.domain.planning_unit import home_unit_id
import gurobipy as gp
from collections import defaultdict
from gurobipy import GRB
from app.domain import SchedulingDataset, PlanningUnitType
from app.domain import POLICY, Assignment, Gap, SchedulingDataset, check_schedule
from app.settings import Settings
from app.solver.models import RunConfiguration, Solution, SolutionStatus, StageReport
from datetime import timedelta
from app.domain import StaffLevel

# --- Testdaten laden ---
from app.solver.bundle import ScheduleInput     # liest input.json der Beispielmonate
import scheduling as fx                         # api/tests/scheduling.py: Bausteine für kleine Testfälle


In [2]:
dataset = ScheduleInput.model_validate_json((ROOT / "plaene" / "2026-06" / "input.json").read_text()).dataset

def only_station(dataset: SchedulingDataset, station_id: int) -> SchedulingDataset:
    """Der Datensatz eingeschränkt auf eine Station: ihr Bedarf und nur die Leute, die dort arbeiten dürfen."""
    # Alle, die an der Station eine Zugehörigkeit haben (Stammpersonal und Springer)
    employees = {m.employee_id for m in dataset.planning_unit_memberships if m.planning_unit_id == station_id}
    # Springerpools bleiben drin: Sie sind die Stammeinheit der Springer, bekommen aber nie Dienste
    pools = {u.planning_unit_id for u in dataset.planning_units if u.type == PlanningUnitType.JUMPER_POOL}
    units = {station_id} | pools

    def keep(rows):
        return tuple(row for row in rows if row.employee_id in employees)

    return SchedulingDataset(
        planning_month=dataset.planning_month,
        planning_units=tuple(u for u in dataset.planning_units if u.planning_unit_id in units),
        shifts=dataset.shifts,
        demand_requirements=tuple(r for r in dataset.demand_requirements if r.planning_unit_id == station_id),
        employees=keep(dataset.employees),
        planning_unit_memberships=tuple(m for m in keep(dataset.planning_unit_memberships) if m.planning_unit_id in units),
        availability=keep(dataset.availability),
        wishes=keep(dataset.wishes),
        monthly_work_accounts=keep(dataset.monthly_work_accounts),
        context=dataset.context.model_copy(update={
            "duties": keep(dataset.context.duties),
            "availability": keep(dataset.context.availability),
        }),
    )

dataset = only_station(dataset, 337)

In [3]:
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

def table(rows, drop=()):
    """Eine Liste von Pydantic-Objekten als DataFrame."""
    return pd.DataFrame([row.model_dump(mode="json", exclude=set(drop)) for row in rows])

print(f"Monat: {dataset.planning_month.label} ({len(dataset.planning_month.dates)} Tage)")
print(f"Stationen: {sorted(dataset.station_ids)}")
for name in ["planning_units", "shifts", "employees", "planning_unit_memberships", "demand_requirements",
             "availability", "wishes", "monthly_work_accounts"]:
    print(f"{name:28s} {len(getattr(dataset, name)):5d} Zeilen")
print(f"{'context.duties':28s} {len(dataset.context.duties):5d} Zeilen  ({dataset.context.covered_from} bis {dataset.context.covered_until})")


Monat: 2026-06 (30 Tage)
Stationen: [337]
planning_units                   3 Zeilen
shifts                           4 Zeilen
employees                       20 Zeilen
planning_unit_memberships       39 Zeilen
demand_requirements            150 Zeilen
availability                    25 Zeilen
wishes                           0 Zeilen
monthly_work_accounts           20 Zeilen
context.duties                  52 Zeilen  (2026-05-27 bis 2026-07-03)


In [4]:
display(table(dataset.planning_units))

def hhmm(minute):
    return f"{minute // 60 % 24:02d}:{minute % 60:02d}" + (" (+1)" if minute >= 1440 else "")

shifts = pd.DataFrame([
    {"shift_id": s.shift_id, "code": s.code, "type": s.type.value,
     "start": hhmm(s.start_minute), "end": hhmm(s.end_minute),
     "segments": ", ".join(f"{hhmm(g.start_minute)}-{hhmm(g.end_minute)}" for g in s.segments),
     "net_work_minutes": s.net_work_minutes}
    for s in dataset.shifts
])
display(shifts)


,planning_unit_id,display_name,type
0,68,PE 68,jumper_pool
1,337,PE 337,station
2,408,PE 408,jumper_pool


,shift_id,code,type,start,end,segments,net_work_minutes
0,1113,F,early,05:55,13:25,"05:55-10:00, 10:30-13:25",420
1,1453,Z,intermediate,08:30,14:15,08:30-14:15,345
2,1605,S,late,13:15,21:00,"13:15-16:00, 16:30-21:00",435
3,1690,N,night,20:10,06:10 (+1),"20:10-21:45, 22:00-00:00 (+1), 00:30 (+1)-06:1...",555


In [5]:
employees = table(dataset.employees)
display(employees)
display(employees["staff_level"].value_counts())

memberships = table(dataset.planning_unit_memberships)
display(memberships)
# Wer darf auf welcher Station mit welcher Qualifikation arbeiten?
display(pd.crosstab(memberships["employee_id"], memberships["planning_unit_id"]))


,employee_id,display_name,staff_level
0,918,Ambriz Waltraud,professional
1,835,Ciaccio Josefa,professional
2,7006,Cormier Sigrid,professional
3,843,Flanagan Mirjam,assistant
4,6318,Gamblin Uschi,professional
5,1167,Min Friedhard,professional
6,839,Säuffert Liselotte,assistant
7,8046,Springer Alina,professional
8,8047,Springer Boris,professional
9,8048,Springer Cem,professional


staff_level
professional    16
assistant        3
trainee          1
Name: count, dtype: int64

,planning_unit_id,employee_id,valid_from,valid_until,staff_level,is_home,is_replacement
0,68,918,2024-10-29,2199-12-31,professional,True,False
1,337,918,2025-12-01,2026-07-31,professional,False,True
2,68,835,2023-04-01,2199-12-31,professional,True,False
3,337,835,2025-12-01,2026-07-31,professional,False,True
4,68,7006,2024-01-01,2199-12-31,professional,True,False
5,337,7006,2025-12-01,2026-07-31,professional,False,True
6,68,843,2023-11-01,2199-12-31,assistant,True,False
7,337,843,2025-12-01,2026-07-31,assistant,False,True
8,68,6318,2023-04-15,2199-12-31,professional,True,False
9,337,6318,2025-12-01,2026-07-31,professional,False,True


planning_unit_id,68,337,408
employee_id,,,
835,1,1,0
839,1,1,0
843,1,1,0
918,1,1,0
1167,0,1,0
1218,1,1,0
5844,1,1,0
6259,1,1,0
6318,1,1,0


In [6]:
demand = table(dataset.demand_requirements)
code = {s.shift_id: s.code for s in dataset.shifts}
demand["shift"] = demand["shift_id"].map(code)
display(demand.head(20))

# Raster je Station: Zeilen = Schicht und Qualifikation, Spalten = Tag
for unit, rows in demand.groupby("planning_unit_id"):
    print(f"Station {unit}")
    display(rows.pivot_table(index=["shift", "staff_level"], columns="date",
                             values="required_count", aggfunc="sum", fill_value=0))


,planning_unit_id,date,shift_id,staff_level,required_count,shift
0,337,2026-06-01,1113,professional,1,F
1,337,2026-06-01,1113,trainee,1,F
2,337,2026-06-01,1605,professional,1,S
3,337,2026-06-01,1605,trainee,1,S
4,337,2026-06-01,1690,professional,1,N
5,337,2026-06-02,1113,professional,1,F
6,337,2026-06-02,1113,trainee,1,F
7,337,2026-06-02,1605,professional,1,S
8,337,2026-06-02,1605,trainee,1,S
9,337,2026-06-02,1690,professional,1,N


Station 337


date                2026-06-01  2026-06-02  2026-06-03  2026-06-04  2026-06-05  2026-06-06  2026-06-07  2026-06-08  2026-06-09  2026-06-10  2026-06-11  2026-06-12  2026-06-13  2026-06-14  \
shift staff_level                                                                                                                                                                            
F     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
      trainee                1           1           1           1           1           1           1           1           1           1           1           1           1           1   
N     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
S     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
      trainee                1           1           1           1           1           1           1           1           1           1           1           1           1           1   

date                2026-06-15  2026-06-16  2026-06-17  2026-06-18  2026-06-19  2026-06-20  2026-06-21  2026-06-22  2026-06-23  2026-06-24  2026-06-25  2026-06-26  2026-06-27  2026-06-28  \
shift staff_level                                                                                                                                                                            
F     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
      trainee                1           1           1           1           1           1           1           1           1           1           1           1           1           1   
N     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
S     professional           1           1           1           1           1           1           1           1           1           1           1           1           1           1   
      trainee                1           1           1           1           1           1           1           1           1           1           1           1           1           1   

date                2026-06-29  2026-06-30  
shift staff_level                           
F     professional           1           1  
      trainee                1           1  
N     professional           1           1  
S     professional           1           1  
      trainee                1           1

In [7]:
accounts = table(dataset.monthly_work_accounts, drop={"credit_details"})
accounts["target_h"] = (accounts["target_minutes"] / 60).round(1)
accounts["open_minutes"] = accounts["target_minutes"] - accounts["credited_minutes"]   # noch zu verplanen
display(accounts)

# Einzelne Gutschriften (Urlaub, Schule, ...)
credits = pd.DataFrame([{"employee_id": a.employee_id, **c.model_dump(mode="json")}
                        for a in dataset.monthly_work_accounts for c in a.credit_details])
display(credits)


,employee_id,target_minutes,actual_minutes,credited_minutes,target_h,open_minutes
0,918,6300,None,0,105.0,6300
1,835,1260,None,0,21.0,1260
2,7006,9702,None,0,161.7,9702
3,843,4284,None,0,71.4,4284
4,6318,9702,None,0,161.7,9702
5,1167,9702,None,0,161.7,9702
6,839,4851,None,0,80.8,4851
7,8046,9702,None,2310,161.7,7392
8,8047,9702,None,2310,161.7,7392
9,8048,9702,None,2310,161.7,7392


,employee_id,date,minutes,kind,source
0,8046,2026-06-08,462,approved_absence,TimeOffice U absence
1,8046,2026-06-09,462,approved_absence,TimeOffice U absence
2,8046,2026-06-10,462,approved_absence,TimeOffice U absence
3,8046,2026-06-11,462,approved_absence,TimeOffice U absence
4,8046,2026-06-12,462,approved_absence,TimeOffice U absence
5,8047,2026-06-15,462,approved_absence,TimeOffice U absence
6,8047,2026-06-16,462,approved_absence,TimeOffice U absence
7,8047,2026-06-17,462,approved_absence,TimeOffice U absence
8,8047,2026-06-18,462,approved_absence,TimeOffice U absence
9,8047,2026-06-19,462,approved_absence,TimeOffice U absence


In [8]:
context = table(dataset.context.duties)
if not context.empty:
    context["shift"] = context["shift_id"].map(code)
    display(context.pivot_table(index="employee_id", columns="date", values="shift",
                                aggfunc="first", fill_value=""))
display(table(dataset.context.availability))


date,2026-05-27,2026-05-28,2026-05-29,2026-05-30,2026-05-31
employee_id,,,,,
839,F,F,,,
918,F,F,,,
1167,F,S,S,N,
5844,S,S,,S,S
6259,F,F,F,,
6318,N,,,,F
6724,,,,Z,N
7006,,F,F,F,S
7490,,F,F,F,F


""


In [9]:
display(pd.Series(POLICY.model_dump(), name="Wert").to_frame())


,Wert
balance_tolerance_minutes,460
min_rest_minutes,660
max_consecutive_nights,3
night_recovery_minutes,2880
max_daily_work_minutes,600
average_daily_work_minutes,480
short_break_after_minutes,360
short_break_minutes,30
long_break_after_minutes,540
long_break_minutes,45


In [10]:
class GurobiSolverService:
    def __init__(self, settings: Settings) -> None:
        self._settings = settings        

    def solve(self, dataset: SchedulingDataset, timeout: float | None = None) -> Solution:
        timeout = timeout if timeout is not None else self._settings.solver_max_time_seconds
        with gp.Env() as env, gp.Model("schedule", env=env) as schedule:
            # Sets and Indices
            U = []
            for unit in dataset.planning_units:
                U.append(unit.planning_unit_id)
            L = [level.value for level in StaffLevel]
            stations_of = {
                e.employee_id: sorted({
                    m.planning_unit_id
                    for m in dataset.planning_unit_memberships
                    if m.employee_id == e.employee_id and m.planning_unit_id in dataset.station_ids
                })
                for e in dataset.employees
            }
            I = []
            I_l = {l:[] for l in L}
            for employee in dataset.employees:
                I.append(employee.employee_id)
                I_l[employee.staff_level].append(employee.employee_id)
            S = []
            S_id = {}
            for shift in dataset.shifts:
                S.append(shift.shift_id)
                S_id[shift.code] = shift.shift_id
            D = list(dataset.planning_month.dates)
            D_prev = [dataset.context.covered_from + timedelta(days=i) for i in range((dataset.planning_month.dates[0] - dataset.context.covered_from).days)]
            D_s = [date for date in dataset.planning_month.dates[:-1] if date.weekday() == 5]
            if D_prev and D_prev[-1].weekday() == 5:
                D_s.append(D_prev[-1])
            # Parameters
            t = {}
            for entry in dataset.monthly_work_accounts:
                t[entry.employee_id] = entry.target_minutes
            h = {}
            for entry in dataset.shifts:
                h[entry.shift_id] = entry.net_work_minutes
            past_shifts = {}
            for entry in dataset.context.duties:
                past_shifts[(entry.employee_id,entry.date)] = entry.shift_id
            L_n = POLICY.night_recovery_minutes//1440
            K_s = 10 # max consecutive workdays
            Dist_n = 7 # days between two nightshift blocks
            R_min = {}
            for s in S:
                for d in D:
                    for u in U:
                        for l in L:
                            R_min[s, d, u, l] = 0
            for entry in dataset.demand_requirements:
                R_min[entry.shift_id,entry.date,entry.planning_unit_id,entry.staff_level] = entry.required_count
            # Decision variables
            assignment = schedule.addVars(I, S, D_prev + D, U, vtype=GRB.BINARY,name="assignment")
            gap = schedule.addVars(S, D, U, L, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="gap")
            n_beg = schedule.addVars(I, D_prev[-Dist_n:] + D, vtype=GRB.CONTINUOUS, lb=0.0, ub=1.0, name="n_beg")
            n_end = schedule.addVars(I, D_prev[-Dist_n:] + D[:-1], vtype=GRB.CONTINUOUS, lb=0.0, ub=1.0, name="n_end")
            hour = schedule.addVars(I, vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY, name="hour")
            hour_max = schedule.addVars(L, vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY, name="hour_max")
            hour_min = schedule.addVars(L, vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY, name="hour_min")
            cons_s = schedule.addVars(I, S, D, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="cons_s")
            cons_x = schedule.addVars(I, D, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="cons_x")
            L_to_E = schedule.addVars(I,D, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="L_to_E")
            # Objective functions
            schedule.setObjectiveN(gap.sum(), index=0, priority=3, name="gaps")
            schedule.setObjectiveN(hour_max.sum() - hour_min.sum(), index=1, priority=2, abstol=300, name="hour_max")
            schedule.setObjectiveN(L_to_E.sum(), index=2, priority=1, name="Min_L_to_E")
            schedule.setObjectiveN(-(cons_s.sum() + 2*cons_x.sum()), index=3, priority=0, name="cons")
            schedule.ModelSense = GRB.MINIMIZE
            # Constraints
            # one shift per day
            schedule.addConstrs((assignment.sum(i, '*', d, '*') <= 1 for i in I for d in D));
            # assign duties
            # assign duties
            for i in I:
                for d in D_prev:
                    if (i,d) in past_shifts.keys():
                        assignment[i, past_shifts[(i,d)],d,U[0]].lb = 1.0
                    else:
                        for s in S:
                            for u in U:
                                assignment[i, s,d, u].ub = 0.0
            # only assign on allowed units
            for i in I:
                for u in U:
                    if u not in stations_of[i]:
                        for s in S:
                            for d in D:
                                assignment[i,s,d,u].ub = 0
            # min demand
            for l in L:
                schedule.addConstrs((assignment.sum(I_l[l], s, d, u) + gap[s, d, u, l] >= R_min[s, d, u, l] for s in S for d in D for u in U));
            # define workhours
            schedule.addConstrs(gp.quicksum(h[s] * assignment.sum(i,s,d,u) for s in S for d in D for u in U) - hour[i] == t[i] for i in I);
            for l in L:
                if I_l[l]:
                    schedule.addConstrs(hour_max[l] >= hour[i] for i in I_l[l]);
                    schedule.addConstrs(hour_min[l] <= hour[i] for i in I_l[l]);
                else:
                    schedule.addConstr(hour_max[l] >= 0)
                    schedule.addConstr(hour_min[l] <= 0)
            # define cons_s and cons_x
            schedule.addConstrs((cons_s[i,s,d] <= assignment.sum(i,s,d,'*') for i in I for s in S for d in D));
            schedule.addConstrs((cons_s[i,s,d] <= assignment.sum(i,s,d-timedelta(days=1),'*') for i in I for s in S for d in D));
            schedule.addConstrs((cons_x[i,d] <= 1 - assignment.sum(i,'*',d,'*') for i in I for d in D));
            schedule.addConstrs((cons_x[i,d] <= 1 - assignment.sum(i,'*',d-timedelta(days=1),'*') for i in I for d in D));
            # max consecutive workdays
            for i in I:
                for d in D_prev[-K_s:] + D[:-K_s]:
                    window = [d+timedelta(days=i) for i in range(K_s + 1)]
                    schedule.addConstr(assignment.sum(i, '*', window, '*') <= K_s)
            # define L_to_E
            schedule.addConstrs(L_to_E[i, d] >= assignment.sum(i, S_id['S'], d-timedelta(days=1),'*') + assignment.sum(i, S_id['F'], d,'*') - 1 for i in I for d in D);
            # max consecutive nights
            for i in I:
                for d in D_prev[-POLICY.max_consecutive_nights:]+ D[:-POLICY.max_consecutive_nights]:
                    window = [d+timedelta(days=i) for i in range(POLICY.max_consecutive_nights + 1)]
                    schedule.addConstr(assignment.sum(i,S_id['N'],window,'*')<= POLICY.max_consecutive_nights)
            # define beginning of nightshiftblock
            schedule.addConstrs((n_beg[i,d] >= assignment.sum(i,S_id['N'],d,'*') - assignment.sum(i,S_id['N'],d-timedelta(days=1),'*') for i in I for d in D_prev[-1:] + D));
            schedule.addConstrs((n_beg[i,d] <= assignment.sum(i,S_id['N'],d,'*') for i in I for d in D_prev[-1:] + D))
            schedule.addConstrs((n_beg[i,d] <= 1 - assignment.sum(i,S_id['N'],d-timedelta(days=1),'*') for i in I for d in D_prev[-1:] + D));
            # define end of nightshiftblock
            schedule.addConstrs((n_end[i,d] >= assignment.sum(i,S_id['N'],d,'*') - assignment.sum(i,S_id['N'],d+timedelta(days=1),'*') for i in I for d in D_prev[-(L_n+1):] + D[:-1]));
            schedule.addConstrs((n_end[i,d] <= assignment.sum(i,S_id['N'],d,'*') for i in I for d in D_prev[-(L_n+1):] + D[:-1]));
            schedule.addConstrs((n_end[i,d] <= 1 - assignment.sum(i,S_id['N'],d+timedelta(days=1),'*') for i in I for d in D_prev[-(L_n+1):] + D[:-1]));
            # no nightshift block starts and ends on the same day
            schedule.addConstrs(n_beg[i,d]+n_end[i,d]<=1 for i in I for d in D_prev[-1:] + D[:-1]);
            # ensure recovery days after nightshift block
            for i in I:
                for d in D_prev[-(L_n+1):] + D[:-1]:
                    for j in range(1,L_n+1):
                        schedule.addConstr(n_end[i, d] + assignment.sum(i, '*', d + timedelta(days=j), '*') <= 1)
                    # no early shift after recovery days
                    schedule.addConstr((n_end[i, d] + assignment.sum(i, S_id['F'], d + timedelta(days=L_n+1), '*') <= 1))
            # same shifts on weekends
            schedule.addConstrs((assignment.sum(i, s, d, '*') == assignment.sum(i, s, d+timedelta(days=1),'*') for i in I for s in S for d in D_s));
            # max 3 weekends each
            schedule.addConstrs((assignment.sum(i, '*', D_s, '*') <= 3 for i in I));
            # min days between nightshift block
            for i in I:
                for d in D_prev[-Dist_n:] + D:
                    forbidden_starts = [d+timedelta(days=j) for j in range(1, Dist_n + 1) if d+timedelta(days=j) in D_prev[-Dist_n:] + D]
                    if forbidden_starts:
                        schedule.addConstr(n_end[i, d] + n_beg.sum(i, forbidden_starts) <= 1)
            # not two cons weekends
            schedule.addConstrs(assignment.sum(i,'*',D_s[j],'*') + assignment.sum(i,'*',D_s[j+1],'*') <= 1 for i in I for j in range(len(D_s) - 1));
            # solve
            schedule.setParam("TimeLimit", timeout)
            schedule.Params.MIPGap = 0.03
            schedule.optimize()
            # construct solution
            assignments = tuple(
                Assignment(employee_id=i, shift_id=s, date=d, planning_unit_id=u, staff_level=l)
                for l in L for i in I_l[l] for s in S for d in D for u in U
                if assignment[i, s, d, u].X > 0.5
            )
            gaps = tuple(
                Gap(shift_id=s, date=d, planning_unit_id=u, staff_level=l, missing_count=int(var.X))
                for (s,d,u,l), var in gap.items()
                if int(var.X) > 0
            )
            stages = []
            for i in range(schedule.NumObj):
                schedule.Params.ObjNumber = i
                stages.append(StageReport(
                    name=schedule.ObjNName,
                    status=SolutionStatus.OPTIMAL if schedule.ObjPassNStatus==2 else SolutionStatus.FEASIBLE,
                    value=int(schedule.ObjNVal),
                    best_bound=int(schedule.ObjPassNObjBound)))
            stages = tuple(stages)
            if schedule.Status == GRB.OPTIMAL:
                status = SolutionStatus.OPTIMAL
            elif schedule.Status in (GRB.INFEASIBLE, GRB.INF_OR_UNBD):
                status = SolutionStatus.INFEASIBLE
            elif schedule.SolCount > 0:
                status = SolutionStatus.FEASIBLE
            else:
                status = SolutionStatus.UNKNOWN

            found = status in (SolutionStatus.OPTIMAL, SolutionStatus.FEASIBLE)
            return Solution(
                status=status,
                configuration=RunConfiguration(policy=POLICY, timeout_seconds=schedule.Params.TimeLimit,
                                            search_workers=None, random_seed=None),
                wall_time_seconds=schedule.Runtime,
                assignments=assignments if found else (),
                gaps=gaps if found else (),
                stages=stages if found else (),
                check=check_schedule(dataset, assignments, gaps) if found else None,
            )

In [11]:
U = []
for unit in dataset.planning_units:
    U.append(unit.planning_unit_id)
L = [level.value for level in StaffLevel]
stations_of = {
    e.employee_id: sorted({
        m.planning_unit_id
        for m in dataset.planning_unit_memberships
        if m.employee_id == e.employee_id and m.planning_unit_id in dataset.station_ids
    })
    for e in dataset.employees
}

In [12]:
I = []
I_l = {l:[] for l in L}
for employee in dataset.employees:
    I.append(employee.employee_id)
    I_l[employee.staff_level].append(employee.employee_id)
S = []
S_id = {}
for shift in dataset.shifts:
    S.append(shift.shift_id)
    S_id[shift.code] = shift.shift_id
D = list(dataset.planning_month.dates)
D_prev = [dataset.context.covered_from + timedelta(days=i) for i in range((dataset.planning_month.dates[0] - dataset.context.covered_from).days)]
D_s = [date for date in dataset.planning_month.dates[:-1] if date.weekday() == 5]
if D_prev and D_prev[-1].weekday() == 5:
    D_s.append(D_prev[-1])

In [13]:
t = {}
credits = {}
for entry in dataset.monthly_work_accounts:
    t[entry.employee_id] = entry.target_minutes
    credits[entry.employee_id] = entry.credited_minutes
h = {}
for entry in dataset.shifts:
    h[entry.shift_id] = entry.net_work_minutes

In [68]:
training = {i:[] for i in I}
absences = {i:{d:0 for d in D_prev[-Dist_n:] + D} for i in I}
for absence in dataset.availability:
    if absence.availability_type == 'training':
        training[absence.employee_id].append(absence.date)
        absences[absence.employee_id,absence.date] = 1
    if absence.availability_type == 'vacation' or absence.availability_type == 'free_day':
        absences[absence.employee_id,absence.date] = 1

In [69]:
past_shifts = {}
for entry in dataset.context.duties:
    past_shifts[(entry.employee_id,entry.date)] = entry.shift_id

In [70]:
L_n = POLICY.night_recovery_minutes//1440
K_s = 10
Dist_n = 7
max_nb = 2

In [71]:
R_min = {}
for s in S:
    for d in D:
        for u in U:
            for l in L:
                R_min[s, d, u, l] = 0
for entry in dataset.demand_requirements:
    R_min[entry.shift_id,entry.date,entry.planning_unit_id,entry.staff_level] = entry.required_count

In [72]:
schedule = gp.Model("schedule")

In [73]:
assignment = schedule.addVars(I, S, D_prev + D, U, vtype=GRB.BINARY,name="assignment")
gap = schedule.addVars(S, D, U, L, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="gap")
n_beg = schedule.addVars(I, D_prev[-Dist_n:] + D, vtype=GRB.CONTINUOUS, lb=0.0, ub=1.0, name="n_beg")
n_end = schedule.addVars(I, D_prev[-Dist_n:] + D[:-1], vtype=GRB.CONTINUOUS, lb=0.0, ub=1.0, name="n_end")
hour = schedule.addVars(I, vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY, name="hour")
hour_max = schedule.addVars(L, vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY, name="hour_max")
hour_min = schedule.addVars(L, vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY, name="hour_min")
cons_s = schedule.addVars(I, S, D, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="cons_s")
cons_x = schedule.addVars(I, D, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="cons_x")
L_to_E = schedule.addVars(I,D, lb=0.0, ub=1.0, vtype=GRB.CONTINUOUS, name="L_to_E")

In [74]:
schedule.setObjectiveN(gap.sum(), index=0, priority=3, name="gaps")
schedule.setObjectiveN(hour_max.sum() - hour_min.sum(), index=1, priority=2, abstol=300, name="hour_max")
schedule.setObjectiveN(L_to_E.sum(), index=2, priority=1, name="Min_L_to_E")
schedule.setObjectiveN(-(cons_s.sum() + 2*cons_x.sum()), index=3, priority=0, name="cons")
schedule.ModelSense = GRB.MINIMIZE

In [75]:
# one shift per day
schedule.addConstrs((assignment.sum(i, '*', d, '*') <= 1 for i in I for d in D));

In [76]:
# assign absences
for absence in dataset.availability:
    if absence.shift_ids is None:
        for s in S:
            for u in U:
                assignment[absence.employee_id,s,absence.date,u].ub = 0.0
    else:
        for s in S:
            if s not in absence.shift_ids:
                for u in U:
                    assignment[absence.employee_id,s,absence.date,u].ub = 0.0

In [77]:
# assign duties
for i in I:
    for d in D_prev:
        if (i,d) in past_shifts.keys():
            assignment[i, past_shifts[(i,d)],d,U[0]].lb = 1.0
        else:
            for s in S:
                for u in U:
                    assignment[i, s,d, u].ub = 0.0

In [78]:
# assign only on allowed units
for i in I:
    for u in U:
        if u not in stations_of[i]:
            for s in S:
                for d in D:
                    assignment[i,s,d,u].ub = 0.0

In [79]:
# min demand
for l in L:
    schedule.addConstrs((assignment.sum(I_l[l], s, d, u) + gap[s, d, u, l] >= R_min[s, d, u, l] for s in S for d in D for u in U));

In [80]:
# define workhours
schedule.addConstrs(gp.quicksum(h[s] * assignment.sum(i,s,d,u) for s in S for d in D for u in U) + credits[i] - hour[i] == t[i] for i in I);
for l in L:
    if I_l[l]:
        schedule.addConstrs(hour_max[l] >= hour[i] for i in I_l[l]);
        schedule.addConstrs(hour_min[l] <= hour[i] for i in I_l[l]);
    else:
        schedule.addConstr(hour_max[l] >= 0)
        schedule.addConstr(hour_min[l] <= 0)

In [81]:
# define cons_s and cons_x
schedule.addConstrs((cons_s[i,s,d] <= assignment.sum(i,s,d,'*') for i in I for s in S for d in D));
schedule.addConstrs((cons_s[i,s,d] <= assignment.sum(i,s,d-timedelta(days=1),'*') for i in I for s in S for d in D));
schedule.addConstrs((cons_x[i,d] <= 1 - assignment.sum(i,'*',d,'*') for i in I for d in D));
schedule.addConstrs((cons_x[i,d] <= 1 - assignment.sum(i,'*',d-timedelta(days=1),'*') for i in I for d in D));

In [82]:
# max consecutive workdays
for i in I:
    for d in D_prev[-K_s:] + D[:-K_s]:
        window = [d+timedelta(days=i) for i in range(K_s + 1)]
        fixed = len(set(window).intersection(set(training[i])))
        schedule.addConstr(assignment.sum(i, '*', window, '*') + fixed <= K_s)

In [83]:
# define L_to_E
schedule.addConstrs(L_to_E[i, d] >= assignment.sum(i, S_id['S'], d-timedelta(days=1),'*') + assignment.sum(i, S_id['F'], d,'*') - 1 for i in I for d in D);

In [84]:
# max consecutive nights
for i in I:
    for d in D_prev[-POLICY.max_consecutive_nights:]+ D[:-POLICY.max_consecutive_nights]:
        window = [d+timedelta(days=i) for i in range(POLICY.max_consecutive_nights + 1)]
        schedule.addConstr(assignment.sum(i,S_id['N'],window,'*')<= POLICY.max_consecutive_nights)

In [85]:
# define beginning of nightshiftblock
schedule.addConstrs((n_beg[i,d] >= assignment.sum(i,S_id['N'],d,'*') - assignment.sum(i,S_id['N'],d-timedelta(days=1),'*') for i in I for d in D_prev[-1:] + D));
schedule.addConstrs((n_beg[i,d] <= assignment.sum(i,S_id['N'],d,'*') for i in I for d in D_prev[-1:] + D))
schedule.addConstrs((n_beg[i,d] <= 1 - assignment.sum(i,S_id['N'],d-timedelta(days=1),'*') for i in I for d in D_prev[-1:] + D));

In [86]:
# define end of nightshiftblock
schedule.addConstrs((n_end[i,d] >= assignment.sum(i,S_id['N'],d,'*') - assignment.sum(i,S_id['N'],d+timedelta(days=1),'*') for i in I for d in D_prev[-(L_n+1):] + D[:-1]));
schedule.addConstrs((n_end[i,d] <= assignment.sum(i,S_id['N'],d,'*') for i in I for d in D_prev[-(L_n+1):] + D[:-1]));
schedule.addConstrs((n_end[i,d] <= 1 - assignment.sum(i,S_id['N'],d+timedelta(days=1),'*') for i in I for d in D_prev[-(L_n+1):] + D[:-1]));

In [87]:
# no nightshift block starts and ends on the same day
schedule.addConstrs(n_beg[i,d]+n_end[i,d]<=1 for i in I for d in D_prev[-1:] + D[:-1]);

In [91]:
# ensure recovery days after nightshift block
for i in I:
    for d in D_prev[-(L_n+1):] + D[:-1]:
        for j in range(1,L_n+1):
            if d + timedelta(days=j) in D_prev[-L_n:] + D:
                schedule.addConstr(n_end[i, d] + assignment.sum(i, '*', d + timedelta(days=j), '*') + absences[i][d+timedelta(days=j)] <= 1)
        # no early shift after recovery days
        schedule.addConstr((n_end[i, d] + assignment.sum(i, S_id['F'], d + timedelta(days=L_n+1), '*') <= 1))


In [92]:
# same shifts on weekends
schedule.addConstrs((assignment.sum(i, s, d, '*') == assignment.sum(i, s, d+timedelta(days=1),'*') for i in I for s in S for d in D_s));

In [93]:
# min days between nightshift block
for i in I:
    for d in D_prev[-Dist_n:] + D:
        forbidden_starts = [d+timedelta(days=j) for j in range(1, Dist_n + 1) if d+timedelta(days=j) in D_prev[-Dist_n:] + D]
        if forbidden_starts:
            schedule.addConstr(n_end[i, d] + n_beg.sum(i, forbidden_starts) <= 1)

In [94]:
# max 3 weekends each
schedule.addConstrs((assignment.sum(i, '*', D_s, '*') <= 3 for i in I));

In [95]:
# not two cons weekends
schedule.addConstrs(assignment.sum(i,'*',D_s[j],'*') + assignment.sum(i,'*',D_s[j+1],'*') <= 1 for i in I for j in range(len(D_s) - 1));

___

In [38]:
schedule.setParam("TimeLimit", 300)
schedule.Params.MIPGap = 0.03
schedule.optimize()

Set parameter TimeLimit to value 300
Set parameter MIPGap to value 0.03
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: Snapdragon(R) X Elite - X1E78100 - Qualcomm(R) Oryon(TM) CPU, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
TimeLimit  300
MIPGap  0.03

Optimize a model with 17182 rows, 14848 columns and 183622 nonzeros (Min)
Model fingerprint: 0x364b291c
Model has 1440 linear objective coefficients
Variable types: 6448 continuous, 8400 integer (8400 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+02]
  Objective range  [1e+00, 2e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+04]


---------------------------------------------------------------------------
Multi-objectives: starting optimization (min) with 4 objectives... 
---------------------------------------------------------------------------

Multi-objectiv

In [49]:
assignments = tuple(
    Assignment(employee_id=i, shift_id=s, date=d, planning_unit_id=u, staff_level=l)
    for l in L for i in I_l[l] for s in S for d in D for u in U
    if assignment[i, s, d, u].X > 0.5
)

In [50]:
gaps = tuple(
    Gap(shift_id=s, date=d, planning_unit_id=u, staff_level=l, missing_count=int(var.X))
    for (s,d,u,l), var in gap.items()
    if int(var.X) > 0
)

In [51]:
stages = []
for i in range(schedule.NumObj):
    schedule.Params.ObjNumber = i
    stages.append(StageReport(
        name=schedule.ObjNName,
        status=SolutionStatus.OPTIMAL if schedule.ObjPassNStatus==2 else SolutionStatus.FEASIBLE,
        value=int(schedule.ObjNVal),
        best_bound=int(schedule.ObjPassNObjBound)))
stages = tuple(stages)

In [52]:
if schedule.Status == GRB.OPTIMAL:
    status = SolutionStatus.OPTIMAL
elif schedule.Status in (GRB.INFEASIBLE, GRB.INF_OR_UNBD):
    status = SolutionStatus.INFEASIBLE
elif schedule.SolCount > 0:
    status = SolutionStatus.FEASIBLE
else:
    status = SolutionStatus.UNKNOWN

found = status in (SolutionStatus.OPTIMAL, SolutionStatus.FEASIBLE)
solution = Solution(
    status=status,
    configuration=RunConfiguration(policy=POLICY, timeout_seconds=schedule.Params.TimeLimit,
                                   search_workers=None, random_seed=None),
    wall_time_seconds=schedule.Runtime,
    assignments=assignments if found else (),
    gaps=gaps if found else (),
    stages=stages if found else (),
    check=check_schedule(dataset, assignments, gaps) if found else None,
)

In [53]:
findings = [f for f in solution.check.findings]

In [54]:
len(findings)

22

In [55]:
findings

[Finding(rule=<Rule.AVAILABILITY: 'availability'>, message='The duty overlaps vacation on 2026-06-22.', employee_id=8048, date=datetime.date(2026, 6, 21), planning_unit_id=337, shift_id=1690),
 Finding(rule=<Rule.AVAILABILITY: 'availability'>, message='A night duty may not precede the approved free day 2026-06-22.', employee_id=8048, date=datetime.date(2026, 6, 21), planning_unit_id=337, shift_id=1690),
 Finding(rule=<Rule.MONTHLY_BALANCE: 'monthly_balance'>, message='Balance -1260 minutes (target 6300, credits 0) is outside ±460.', employee_id=918, date=None, planning_unit_id=None, shift_id=None),
 Finding(rule=<Rule.MONTHLY_BALANCE: 'monthly_balance'>, message='Balance -1260 minutes (target 1260, credits 0) is outside ±460.', employee_id=835, date=None, planning_unit_id=None, shift_id=None),
 Finding(rule=<Rule.MONTHLY_BALANCE: 'monthly_balance'>, message='Balance -1182 minutes (target 9702, credits 0) is outside ±460.', employee_id=7006, date=None, planning_unit_id=None, shift_id=No

In [56]:
import platform
from hashlib import sha256
from pathlib import Path

import pydantic
from app.solver.bundle import InputReference, RuntimeVersions, ScheduleInput, ScheduleResult, to_json

out = Path("export")
out.mkdir(exist_ok=True)

input_json = to_json(ScheduleInput.of(dataset))
result = ScheduleResult(
    format_version=3,
    planning_month=dataset.planning_month,
    input=InputReference(file="input.json", sha256=sha256(input_json).hexdigest()),
    solution=solution,
    versions=RuntimeVersions(python=platform.python_version(), ortools="gurobi", pydantic=pydantic.VERSION),
)
(out / "input.json").write_bytes(input_json)
(out / "result.json").write_bytes(to_json(result))


52687

In [57]:
dataset.availability

(Availability(availability_type=<AvailabilityType.VACATION: 'vacation'>, reason='U', shift_ids=None, employee_id=8046, date=datetime.date(2026, 6, 8), native_absence=True),
 Availability(availability_type=<AvailabilityType.VACATION: 'vacation'>, reason='U', shift_ids=None, employee_id=8046, date=datetime.date(2026, 6, 9), native_absence=True),
 Availability(availability_type=<AvailabilityType.VACATION: 'vacation'>, reason='U', shift_ids=None, employee_id=8046, date=datetime.date(2026, 6, 10), native_absence=True),
 Availability(availability_type=<AvailabilityType.VACATION: 'vacation'>, reason='U', shift_ids=None, employee_id=8046, date=datetime.date(2026, 6, 11), native_absence=True),
 Availability(availability_type=<AvailabilityType.VACATION: 'vacation'>, reason='U', shift_ids=None, employee_id=8046, date=datetime.date(2026, 6, 12), native_absence=True),
 Availability(availability_type=<AvailabilityType.VACATION: 'vacation'>, reason='U', shift_ids=None, employee_id=8047, date=datetime